# 03 · Inference Pipeline

Demonstrates how to use the trained model to generate predictions on new data.
Reproduces the full preprocessing + prediction pipeline from raw processed data to final output.

**Required artefacts** (generated by `02_modeling.ipynb`):

| File | Contents |
|------|----------|
| `models/model_tuned.pkl` | Tuned XGBoost classifier |
| `models/iso_reg.pkl` | Isotonic regression calibrator |
| `models/encoders.pkl` | Target encoders, OHE, feature metadata |
| `models/threshold.pkl` | Optimal decision threshold (recall ≥ 0.80) |

**Expected input:** a DataFrame with the same columns as `data_processed.csv`,
without `poor_control` or `year` (not available at inference time).

## 1. Load artefacts

In [10]:
import pandas as pd
import numpy as np
import joblib
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, classification_report

model     = joblib.load('models/model_tuned.pkl')
iso_reg   = joblib.load('models/iso_reg.pkl')
encoders  = joblib.load('models/encoders.pkl')
threshold = joblib.load('models/threshold.pkl')

target_encoders   = encoders['target_encoders']
ohe               = encoders['ohe']
global_mean       = encoders['global_mean']
ohe_cols          = encoders['ohe_cols']
drop_features     = encoders['drop_features']
expected_features = encoders['features']

print(f'Model:              {type(model).__name__}')
print(f'Decision threshold: {threshold:.3f}')
print(f'Expected features:  {len(expected_features)}')

Model:              XGBClassifier
Decision threshold: 0.139
Expected features:  47


## 2. Preprocessing function

Encapsulates the full transformation pipeline in a single reusable function.
Accepts a DataFrame matching the structure of `data_processed.csv` and returns
a feature matrix ready for `model.predict_proba()`.

In [11]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform a processed DataFrame (output of EDA pipeline) into the
    feature matrix expected by the model.

    Steps:
        1. Drop non-feature columns (target, year)
        2. Target encoding for area and zona_ap
        3. One-hot encoding for eos_nivel, ige_nivel and their lags
        4. Drop zero-importance features (identified via SHAP)
        5. Reorder columns to match training feature order
    """
    X = df.drop(columns=['poor_control', 'year'], errors='ignore').copy()

    # Target encoding (unseen categories → global mean)
    for col, mapping in target_encoders.items():
        X[col] = X[col].map(mapping).fillna(global_mean)

    # One-hot encoding
    encoded    = ohe.transform(X[ohe_cols])
    encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(ohe_cols), index=X.index)
    X = pd.concat([X.drop(columns=ohe_cols), encoded_df], axis=1)

    # Drop zero-importance features
    X = X.drop(columns=[c for c in drop_features if c in X.columns])

    # Align to training feature order
    return X[expected_features]


## 3. Prediction function

Chains preprocessing → XGBoost → isotonic calibration into a single call.
Returns binary predictions and, optionally, calibrated probabilities.

In [12]:
def predict(df: pd.DataFrame, return_proba: bool = False):
    """
    Generate poor-control predictions for a DataFrame of patient-year records.

    Args:
        df:           Processed DataFrame (output of EDA pipeline)
        return_proba: if True, also return calibrated probability scores

    Returns:
        preds:  Series[int]   — 0 = good control, 1 = poor control
        probas: Series[float] — calibrated probability of poor control
                                (only returned if return_proba=True)
    """
    X         = preprocess(df)
    proba_raw = model.predict_proba(X)[:, 1]
    proba_cal = iso_reg.predict(proba_raw)
    preds     = (proba_cal >= threshold).astype(int)

    preds_s = pd.Series(preds,     index=df.index, name='poor_control_pred')
    proba_s = pd.Series(proba_cal, index=df.index, name='poor_control_proba')

    return (preds_s, proba_s) if return_proba else preds_s


## 4. Example: predictions on 2024 cohort

The 2024 test set is used as a stand-in for new incoming data.
In a real deployment scenario, this would be a fresh annual extract from the EHR system.

In [13]:
df       = pd.read_csv('data_processed.csv')
new_data = df[df['year'] == 2024].copy()

print(f'Patients to score: {len(new_data):,}')
new_data.head(3)

Patients to score: 48,105


,area,ap_zone,sex,year,age,arterial_hypertension,chronic_kidney_disease,cardiovascular_disease,obesity,heart_failure,...,biologic_treatment_lag1,pulmonology_follow_up_lag1,allergy_follow_up_lag1,spirometry_lag1,prick_test_lag1,feno_lag1,eos_level_lag1,ige_level_lag1,poor_control_acum,years_in_record
6,area_santander,el_alisal,0,2024,67,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,normal,no_analitics,0,6
13,area_laredo,santońa,1,2024,66,1,0,0,1,0,...,0.0,0.0,1.0,0.0,0.0,0.0,moderate,no_analitics,0,6
14,area_santander,centro,0,2024,45,0,0,0,1,0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,no_history,no_history,-1,0


In [14]:
preds, probas = predict(new_data, return_proba=True)

results = new_data[['year']].copy()
results['poor_control_actual'] = new_data['poor_control'].values
results['poor_control_proba']  = probas.values
results['poor_control_pred']   = preds.values

print('Prediction distribution:')
print(results['poor_control_pred'].value_counts())
print()
results.head(10)

Prediction distribution:
poor_control_pred
0    27114
1    20991
Name: count, dtype: int64



,year,poor_control_actual,poor_control_proba,poor_control_pred
6,2024,0,0.288000,1
13,2024,0,0.320158,1
14,2024,0,0.434951,1
17,2024,1,0.222074,1
24,2024,0,0.098276,0
35,2024,0,0.351384,1
42,2024,0,0.041728,0
46,2024,1,0.627451,1
50,2024,0,0.018796,0
63,2024,0,0.138166,0


In [15]:
y_true = results['poor_control_actual']
y_pred = results['poor_control_pred']
y_prob = results['poor_control_proba']

print(f'ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}\n')
print(classification_report(y_true, y_pred, target_names=['good control', 'poor control']))

ROC-AUC: 0.8192

              precision    recall  f1-score   support

good control       0.94      0.65      0.77     39491
poor control       0.34      0.83      0.48      8614

    accuracy                           0.68     48105
   macro avg       0.64      0.74      0.63     48105
weighted avg       0.84      0.68      0.72     48105



## 5. Output interpretation

| Field | Description |
|-------|-------------|
| `poor_control_proba` | Calibrated probability of poor control in the next year (0–1). Directly interpretable: 0.30 means ~30% real risk. |
| `poor_control_pred` | Binary classification using threshold ≈ 0.157, optimised for recall ≥ 0.80. |

### Clinical threshold rationale

The threshold was chosen to **prioritise recall** over precision, reflecting the asymmetric
cost structure of the clinical problem:

| Error type | Clinical consequence | Cost |
|------------|----------------------|------|
| False negative | Exacerbation goes undetected → A&E / hospitalisation | **High** |
| False positive | Unnecessary follow-up visit | Low |

At the chosen threshold the model detects ~80% of poor-control patients (recall),
at the cost of ~65% false positives among those flagged (precision ~0.35).
This is an acceptable trade-off for an early-warning screening tool.